In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.utils import class_weight
import os

# Paths for data
data_train_path = 'D:/Lehan/dataset cataract/archive/dataset/newmodel/new_train_data/train'
data_val_path = 'D:/Lehan/dataset cataract/archive/dataset/newmodel/new_train_data/validation'

# Image dimensions
img_width, img_height = 180, 180
batch_size = 32
epochs_size = 30

# Load datasets
data_train = tf.keras.utils.image_dataset_from_directory(
    data_train_path,
    shuffle=True,
    image_size=(img_width, img_height),
    batch_size=batch_size
)
data_val = tf.keras.utils.image_dataset_from_directory(
    data_val_path,
    shuffle=False,
    image_size=(img_width, img_height),
    batch_size=batch_size
)

# Get class names
data_cat = data_train.class_names

# Calculate class weights to address imbalance
labels = np.concatenate([y for x, y in data_train], axis=0)
class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights_dict = dict(enumerate(class_weights))
print("Class weights:", class_weights_dict)

# Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

# Define the model architecture with data augmentation
model = Sequential([
    data_augmentation,
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(data_cat), activation='softmax')  # softmax activation for multi-class classification
])


# Compile the model
model.compile(optimizer='adam', 
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
              metrics=['accuracy'])

# Add a checkpoint to save the model with the best validation accuracy
checkpoint = ModelCheckpoint('best_model.keras', 
                             monitor='val_accuracy', 
                             mode='max', 
                             save_best_only=True, 
                             verbose=1)


# Train the model with class weights and validation accuracy monitoring
history = model.fit(
    data_train,
    validation_data=data_val,
    epochs=epochs_size,
    class_weight=class_weights_dict,
    callbacks=[checkpoint]
)

# Evaluate the model
val_loss, val_accuracy = model.evaluate(data_val)
print(f"Final validation accuracy: {val_accuracy:.2f}")


Found 2889 files belonging to 4 classes.
Found 1785 files belonging to 4 classes.
Class weights: {0: 1.1320532915360502, 1: 1.1898682042833608, 2: 0.8263729977116705, 3: 0.937987012987013}
Epoch 1/30
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.3305 - loss: 1.4167
Epoch 1: val_accuracy improved from -inf to 0.42129, saving model to best_model.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 16s 159ms/step - accuracy: 0.3315 - loss: 1.4147 - val_accuracy: 0.4213 - val_loss: 1.2220
Epoch 2/30
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - accuracy: 0.4660 - loss: 1.1940
Epoch 2: val_accuracy did not improve from 0.42129
91/91 ━━━━━━━━━━━━━━━━━━━━ 15s 161ms/step - accuracy: 0.4665 - loss: 1.1935 - val_accuracy: 0.4179 - val_loss: 1.2463
Epoch 3/30
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.5140 - loss: 1.1143
Epoch 3: val_accuracy improved from 0.42129 to 0.42297, saving model to best_model.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 16s 171ms/step - accuracy: 0.5140 - loss: 1.1145 - val_accuracy:

In [17]:
# Load the best saved model for inference
loaded_model = tf.keras.models.load_model('best_model.keras')

# Function to preprocess and predict on a new image
def predict_image(model, image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(img_height, img_width))
    img_array = tf.keras.utils.img_to_array(img) / 255.0  # Normalize as done in training
    img_bat = np.expand_dims(img_array, axis=0)  # Add batch dimension

    predictions = model.predict(img_bat)
    score = tf.nn.softmax(predictions[0])  # Softmax to get probabilities
    predicted_class = data_cat[np.argmax(score)]
    confidence = np.max(score) * 100
    predictions = model.predict(data_val)  # Get the raw predictions
    predicted_classes = np.argmax(predictions, axis=1)  # Get the class with the highest probability
    predicted_probabilities = np.max(predictions, axis=1)  # Get the probabilities for the predicted classes
    print(predicted_classes, predicted_probabilities)


    print(f"Predicted class: {predicted_class}, Confidence: {confidence:.2f}%")

# Test prediction on an example image
image_path = "D:/Lehan/dataset cataract/archive/dataset/newmodel/new_train_data/train/cataractT/_256_5928804.jpg"
predict_image(loaded_model, image_path)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
56/56 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step
[0 0 0 ... 2 1 2] [1.1935524 2.0671234 2.3230877 ... 1.2794753 2.101628  1.6503756]
Predicted class: retina_diseaseT, Confidence: 50.02%
